# Gen AI Gateway with Azure APIM

This lab demonstrates several capabilities of Azure API Management service acting as a Gen AI Gateway.

The **Setup** section must be run before executing subsequent labs. Labs are independent and can be run in any order.

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment with uv](https://docs.astral.sh/uv/pip/environments/)
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`... 

## Setup

<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string to ensure resource uniqueness.

In [ ]:
import sys
sys.path.insert(1, './shared')  # add the shared directory to the Python path
import utils
import random, string

# Generate a dynamic unique 4-character suffix (letters + digits)
unique_name = ''.join(random.choices(string.ascii_lowercase + string.digits, k=4))
%store unique_name

utils.print_ok('Notebook initialized. Suffix: ' + unique_name)

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Provision infrastructure using 🚜 Terraform

We now deploy the lab resources (API Management, Redis cache, AI Foundry account, model deployments, etc.) with **Terraform**.

Steps performed by the next cell:

1. Run `terraform init` (upgrade providers if needed).
1. Run `terraform apply` passing the `unique_name` variable.

In [ ]:
terraform_dir = 'terraform'

# Initialize Terraform (upgrade providers to ensure latest versions)
init_output = utils.run(f"terraform -chdir={terraform_dir} init -upgrade", "Terraform init succeeded", "Terraform init failed")

if init_output.success:
    apply_output = utils.run(
        f"ARM_SUBSCRIPTION_ID={subscription_id} terraform -chdir={terraform_dir} apply -auto-approve -var unique_name={unique_name}",
        "Terraform apply succeeded",
        "Terraform apply failed",
        print_output=True
    )
    if apply_output.success:
        utils.print_ok("Infrastructure provisioned via Terraform.")
    else:
        utils.print_error("Terraform apply failed. Scroll above for details.")
else:
    utils.print_error("Terraform init failed. Fix issues and re-run this cell.")